# 🎯 Semantic Watermark Inference Pipeline

## 📌 Overview

이 노트북은 학습된 VAE 모델과 VINE을 사용하여 이미지에 semantic watermark를 삽입하고,
편집된 이미지에서 워터마크를 추출하여 semantic 정보를 복원하는 전체 파이프라인을 실행합니다.

### **Pipeline 구조**
```
1. Input Image → CLIP Embedding → VAE Encoder → Latent (100D)
2. Latent → Watermark (100-bit) → VINE Encoder → Watermarked Image
3. Watermarked Image → External Edit (InstructPix2Pix 등) → Edited Image
4. Edited Image → VINE Decoder → Watermark (100-bit) → Latent (100D)
5. Latent → VAE Decoder → Reconstructed CLIP Embedding
6. Analysis: Original CLIP vs Reconstructed CLIP (Semantic Preservation)
```

### **Directory Structure**
```
/content/drive/MyDrive/semantic_wm/
├── models/
│   ├── best_vae_model.pth          # 학습된 VAE 모델
│   └── best_vqvae_model.pth        # 학습된 VQ-VAE 모델
└── inference_dataset/
    ├── input/                      # 원본 이미지
    ├── output/                     # 워터마크 삽입된 이미지
    └── edited/                     # 외부에서 편집된 이미지
```

### **Key Features**
- ✅ Watermark 삽입 (VINE-B-Enc)
- ✅ Watermark 추출 (VINE-B-Dec)
- ✅ Semantic 정보 복원 (VAE Decoder)
- ✅ 카테고리별 정량 분석
- ✅ 다양한 시각화 (Bar, Line, t-SNE)

## 1. 패키지 설치 및 Import

In [ ]:
# PyTorch 및 기본 라이브러리 설치
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy scipy pillow opencv-python
!pip install -q scikit-learn scikit-image matplotlib seaborn pandas
!pip install -q tqdm einops timm

print("✅ 기본 패키지 설치 완료")

In [ ]:
# CLIP 및 Transformers 설치
!pip install -q transformers==4.45.2 open-clip-torch==2.26.1

print("✅ CLIP 패키지 설치 완료")

In [ ]:
# Diffusers (InstructPix2Pix용)
!pip install -q diffusers accelerate

print("✅ Diffusers 패키지 설치 완료")

In [ ]:
!pip install --upgrade transformers diffusers accelerate -q

In [ ]:
# 런타임 재시작
import os
os.kill(os.getpid(), 9)

In [ ]:
# 기본 라이브러리 Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from tqdm import tqdm
import time
import gc
import warnings
import sys
import os
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Sklearn
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("✅ 기본 라이브러리 Import 완료")
print("="*80)

## 2. 환경 설정 및 경로 확인

In [ ]:
# Google Drive 마운트
from google.colab import drive

print("="*80)
print("Google Drive 마운트 중...")
print("="*80)

drive.mount('/content/drive')

print("\n✅ Google Drive 마운트 완료")

In [ ]:
# semantic_wm 데이터셋 압축 해제
zip_path = '/content/drive/MyDrive/semantic_wm/dataset.zip'

print("압축 해제 중...")
!unzip -q {zip_path} -d /content/semantic_wm/
print("✅ 데이터셋 압축 해제 완료")

# 데이터셋 구조 확인
print("\n=== 데이터셋 구조 ===")
!ls -R /content/semantic_wm/dataset/train/

print("\n각 카테고리별 이미지 개수:")
for category in ['normal', 'sexual', 'violence']:
    category_path = f'/content/semantic_wm/dataset/train/{category}'
    if os.path.exists(category_path):
        count = len([f for f in os.listdir(category_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
        print(f"  - {category}: {count}장")
    else:
        print(f"  - {category}: 경로 없음")

print("="*80)

In [ ]:
# 경로 설정
BASE_DIR = '/content/drive/MyDrive/semantic_wm'
MODEL_DIR = os.path.join(BASE_DIR, 'models')
DATASET_DIR = os.path.join(BASE_DIR, 'inference_dataset')
INPUT_DIR = os.path.join(DATASET_DIR, 'input')
OUTPUT_DIR = os.path.join(DATASET_DIR, 'output')
EDITED_DIR = os.path.join(DATASET_DIR, 'edited')

# Training 데이터셋 경로 (압축 해제된 위치)
TRAINING_DATA_DIR = '/content/semantic_wm/dataset/train'

print("="*80)
print("경로 설정")
print("="*80)
print(f"BASE_DIR:          {BASE_DIR}")
print(f"MODEL_DIR:         {MODEL_DIR}")
print(f"INPUT_DIR:         {INPUT_DIR}")
print(f"OUTPUT_DIR:        {OUTPUT_DIR}")
print(f"EDITED_DIR:        {EDITED_DIR}")
print(f"TRAINING_DATA_DIR: {TRAINING_DATA_DIR}")
print("="*80)

In [ ]:
# 디렉토리 존재 확인 및 생성
print("\n" + "="*80)
print("디렉토리 확인 및 생성")
print("="*80)

directories = {
    'BASE': BASE_DIR,
    'MODELS': MODEL_DIR,
    'DATASET': DATASET_DIR,
    'INPUT': INPUT_DIR,
    'OUTPUT': OUTPUT_DIR,
    'EDITED': EDITED_DIR
}

for name, path in directories.items():
    if os.path.exists(path):
        print(f"✓ {name:<10} 존재함: {path}")
    else:
        os.makedirs(path, exist_ok=True)
        print(f"✓ {name:<10} 생성함: {path}")

print("="*80)

In [ ]:
# 파일 목록 확인
import glob

print("\n" + "="*80)
print("파일 목록 확인")
print("="*80)

# Models
model_files = glob.glob(os.path.join(MODEL_DIR, '*.pth'))
print(f"\n[Models Directory - {len(model_files)} files]")
for f in model_files:
    print(f"  - {os.path.basename(f)}")

# Input images
input_images = glob.glob(os.path.join(INPUT_DIR, '*.*'))
print(f"\n[Input Directory - {len(input_images)} files]")
for f in input_images[:10]:  # 최대 10개만 표시
    print(f"  - {os.path.basename(f)}")
if len(input_images) > 10:
    print(f"  ... and {len(input_images) - 10} more files")

# Output images
output_images = glob.glob(os.path.join(OUTPUT_DIR, '*.*'))
print(f"\n[Output Directory - {len(output_images)} files]")
for f in output_images[:10]:
    print(f"  - {os.path.basename(f)}")
if len(output_images) > 10:
    print(f"  ... and {len(output_images) - 10} more files")

# Edited images
edited_images = glob.glob(os.path.join(EDITED_DIR, '*'))
print(f"\n[Edited Directory - {len(edited_images)} files]")
for f in edited_images[:10]:
    print(f"  - {os.path.basename(f)}")
if len(edited_images) > 10:
    print(f"  ... and {len(edited_images) - 10} more files")

print("="*80)
print("✅ 디렉토리 구조 확인 완료")
print("="*80)

In [ ]:
print("="*80)
print("VINE Repository 클론 중...")
print("="*80)

# VINE이 이미 클론되어 있는지 확인
if not os.path.exists('/content/VINE'):
    !git clone https://github.com/Shilin-LU/VINE.git /content/VINE
    print("✅ VINE 클론 완료")
else:
    print("✅ VINE이 이미 존재함")

# Python path에 추가
sys.path.append("/content/VINE")

print("="*80)

In [ ]:
# CLIP 모델 Import
from transformers import CLIPProcessor, CLIPModel

print("✅ CLIP Import 완료")

In [ ]:
# VINE 모델 Import
from vine.src.vine_turbo import VINE_Turbo
from vine.src.stega_encoder_decoder import CustomConvNeXt
from accelerate.utils import set_seed

print("✅ VINE Import 완료")

In [ ]:
# Diffusers Import (InstructPix2Pix)
from diffusers import StableDiffusionInstructPix2PixPipeline, DDIMScheduler

print("✅ Diffusers Import 완료")

In [ ]:
# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("Device 정보")
print("="*80)
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
else:
    print("⚠️ CUDA를 사용할 수 없습니다. CPU로 실행됩니다.")

print("="*80)

# Random seed 설정
set_seed(42)
torch.manual_seed(42)
np.random.seed(42)

print("✅ Random seed 설정 완료 (seed=42)")

## 3. 모델 로딩 (VAE, CLIP, VINE)

In [ ]:
# VAE 모델 아키텍처 정의 (nn.Sequential 방식)
class ImprovedVAE(nn.Module):
    """
    Improved VAE for CLIP Embedding Compression
    - Input: 512D CLIP embedding
    - Latent: 100D compressed representation
    - Output: 512D reconstructed CLIP embedding (L2 normalized)

    개선사항:
    - BatchNorm1d 사용 (LayerNorm 대신)
    - Decoder 출력에 L2 정규화 적용
    - Dropout 0.2 (더 강한 정규화)
    """
    def __init__(self, input_dim=512, latent_dim=100):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Encoder: CLIP(512) → Latent(latent_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder: Latent(latent_dim) → CLIP'(512)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        """Encoder: CLIP → Latent (μ, σ)"""
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = μ + σ * ε"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        """Decoder: Latent → CLIP (L2 normalized)"""
        recon = self.decoder(z)
        # ✅ L2 정규화 적용 (중요!)
        return F.normalize(recon, p=2, dim=1)

    def forward(self, x):
        """Forward: CLIP → Latent → Reconstructed CLIP"""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

    def get_latent(self, x):
        """Inference용: latent 추출 (μ only, no sampling)"""
        mu, _ = self.encode(x)
        return mu

print("="*80)
print("✅ VAE 모델 아키텍처 정의 완료")
print("="*80)
print(f"   - Input: {512}D CLIP embedding")
print(f"   - Latent: {100}D compressed representation")
print(f"   - Output: {512}D reconstructed CLIP (L2 normalized)")
print(f"   - Architecture: nn.Sequential (BatchNorm1d + Dropout 0.2)")
print("="*80)

In [ ]:
# VAE 모델 로드
print("="*80)
print("VAE 모델 로드 중...")
print("="*80)

vae_model = ImprovedVAE(input_dim=512, latent_dim=100).to(device)

# 학습된 가중치 로드
vae_model_path = os.path.join(MODEL_DIR, 'best_vae_model.pth')

if os.path.exists(vae_model_path):
    vae_model.load_state_dict(torch.load(vae_model_path, map_location=device))
    vae_model.eval()
    print(f"✅ VAE 모델 로드 완료")
    print(f"   - Path: {vae_model_path}")
    print(f"   - Parameters: {sum(p.numel() for p in vae_model.parameters()):,}")
else:
    print(f"❌ VAE 모델을 찾을 수 없습니다: {vae_model_path}")
    print(f"   먼저 vae_vqvae_comparison_improved.ipynb에서 모델을 학습해주세요.")

print("="*80)

In [ ]:
# CLIP 모델 로드
print("\\n" + "="*80)
print("CLIP 모델 로드 중...")
print("="*80)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

print("✅ CLIP 모델 로드 완료")
print(f"   - Model: openai/clip-vit-base-patch32")
print(f"   - Embedding dimension: 512D")
print("="*80)

In [ ]:
# VINE 모델 로드
print("="*80)
print("VINE 모델 로드 중...")
print("="*80)

# VINE Encoder (Watermark Embedding)
watermark_encoder = VINE_Turbo.from_pretrained("Shilin-LU/VINE-B-Enc").to(device)
watermark_encoder.eval()

# VINE Decoder (Watermark Extraction)
vine_decoder = CustomConvNeXt.from_pretrained("Shilin-LU/VINE-B-Dec").to(device)
vine_decoder.eval()

print("✅ VINE 모델 로드 완료")
print(f"   - Encoder: VINE-B-Enc (100-bit watermark embedding)")
print(f"   - Decoder: VINE-B-Dec (watermark extraction)")
print("="*80)

In [ ]:
# 모델 요약
print("="*80)
print("모든 모델 로드 완료 - 요약")
print("="*80)

models_info = [
    ("VAE Encoder/Decoder", "512D → 100D → 512D", "Semantic compression"),
    ("CLIP", "Image → 512D embedding", "Feature extraction"),
    ("VINE Encoder", "Image + 100-bit → Watermarked Image", "Watermark embedding"),
    ("VINE Decoder", "Watermarked Image → 100-bit", "Watermark extraction")
]

for name, spec, desc in models_info:
    print(f"✓ {name}")
    print(f"  ├─ Spec: {spec}")
    print(f"  └─ Role: {desc}")

print("="*80)

# GPU 메모리 사용량 확인
if torch.cuda.is_available():
    print(f"[GPU Memory Usage]")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print("="*80)

## 4. Watermarker 클래스 정의

In [ ]:
class LatentWatermarker:
    """
    Latent Space Watermarking System with Statistical Latent Restoration

    Pipeline:
        1. Image → CLIP → 512D embedding
        2. CLIP → VAE Encoder → 100D latent
        3. Latent → Sign-bit quantization → 100-bit watermark
        4. Watermark → {-1,+1} → (mean,std) scaling → 100D latent
        5. Latent → VAE Decoder → 512D reconstructed CLIP
    """

    def __init__(self, vae_model, device='cuda'):
        self.vae_model = vae_model
        self.device = device
        self.vae_model.eval()

    # ------------------------------------------
    # Latent → Watermark
    # ------------------------------------------
    def latent_to_watermark(self, latent):
        """
        Convert latent vector to 100-bit binary watermark (sign-bit)
        latent > 0 → 1
        latent ≤ 0 → 0
        """
        return (latent > 0).float()

    # ------------------------------------------
    # Watermark → Latent (with optional scaling)
    # ------------------------------------------
    def watermark_to_latent(self, watermark, latent_stats=None):
        """
        Convert 100-bit binary watermark → restored latent vector.
        Step 1: 0/1 → -1/+1
        Step 2: (optional) scale back using latent mean/std

        Args:
            watermark: [batch, 100] or [100]
            latent_stats: (mean, std), each [100]

        Returns:
            latent_restored: [batch, 100]
        """
        # 0→-1, 1→+1
        latent_approx = 2.0 * watermark - 1.0

        # 통계 기반 스케일 복원
        if latent_stats is not None:
            mean, std = latent_stats  # 각각 shape [100]

            # Ensure tensor
            if not torch.is_tensor(mean):
                mean = torch.tensor(mean, device=self.device, dtype=latent_approx.dtype)
            if not torch.is_tensor(std):
                std = torch.tensor(std, device=self.device, dtype=latent_approx.dtype)

            # broadcast-safe scaling
            latent_approx = latent_approx * std + mean

        return latent_approx

    # ------------------------------------------
    # CLIP Embedding → Watermark
    # ------------------------------------------
    def embed_watermark(self, clip_embedding):
        """
        Convert CLIP embedding → latent → binary watermark
        Returns:
            watermark: [batch, 100]
            original_latent: [batch, 100]
        """
        with torch.no_grad():
            original_latent = self.vae_model.get_latent(clip_embedding)
            watermark = self.latent_to_watermark(original_latent)
        return watermark, original_latent

    # ------------------------------------------
    # Watermark → CLIP reconstruction
    # ------------------------------------------
    def extract_and_reconstruct(self, watermark, latent_stats=None):
        """
        Convert watermark → latent → reconstructed CLIP

        Args:
            watermark: [batch, 100]
            latent_stats: (mean, std)

        Returns:
            reconstructed_clip: [batch, 512]
            reconstructed_latent: [batch, 100]
        """
        with torch.no_grad():
            reconstructed_latent = self.watermark_to_latent(watermark, latent_stats)
            reconstructed_clip = self.vae_model.decode(reconstructed_latent)

        return reconstructed_clip, reconstructed_latent

print("="*80)
print("✅ LatentWatermarker 클래스 정의 완료 (통계 기반 복원)")
print("="*80)
print("주요 메서드:")
print("  • embed_watermark(clip_embedding)")
print("    → CLIP (512D) → VAE Encoder → Latent (100D) → Watermark (100-bit)")
print("  • extract_and_reconstruct(watermark, latent_stats=None)")
print("    → Watermark (100-bit) → Latent (100D) → VAE Decoder → CLIP (512D)")
print("    → latent_stats=(mean, std)로 통계 기반 복원 가능")
print("  • latent_to_watermark(latent)")
print("    → Sign-bit quantization: latent > 0 → 1, latent ≤ 0 → 0")
print("  • watermark_to_latent(watermark, latent_stats=None)")
print("    → Binary mapping: 0 → -1.0, 1 → +1.0")
print("    → 통계 복원: latent' = latent * std + mean")
print("="*80)

In [ ]:
# Watermarker 인스턴스 생성
latent_watermarker = LatentWatermarker(vae_model=vae_model, device=device)

print("="*80)
print("✅ LatentWatermarker 인스턴스 생성 완료")
print("="*80)
print("준비 완료:")
print("  ✓ VAE 모델 연결")
print("  ✓ Sign-bit quantization 방식")
print("  ✓ 100D latent ↔ 100-bit watermark 변환")
print("="*80)

## 5. 워터마크 삽입 (Input → Output)

In [ ]:
def process_watermark_insertion(input_dir, output_dir, clip_model, clip_processor,
                                 watermark_encoder, latent_watermarker, device='cuda'):
    """
    Insert semantic watermark using VINE encoder.
    각 이미지의 원본 CLIP embedding과 latent 통계를 함께 저장합니다.
    """

    # 이미지 파일 목록
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    image_files = [f for f in os.listdir(input_dir)
                   if f.lower().endswith(image_extensions)]

    if len(image_files) == 0:
        print(f"❌ {input_dir}에서 이미지를 찾을 수 없습니다.")
        return []

    print(f"발견된 이미지: {len(image_files)}장")
    print("="*80)

    # VINE 입력 전처리기
    to256 = transforms.Compose([
        transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2.0 - 1.0),   # [0,1] → [-1,1]
    ])

    results = []

    for idx, filename in enumerate(tqdm(image_files, desc="워터마크 삽입 중")):
        try:
            # 1. 이미지 로드
            input_path = os.path.join(input_dir, filename)
            image = Image.open(input_path).convert('RGB')

            # 2. CLIP 임베딩 추출 (원본)
            inputs = clip_processor(images=image, return_tensors="pt", padding=True)
            pixel_values = inputs['pixel_values'].to(device)

            with torch.no_grad():
                original_clip_embedding = clip_model.get_image_features(pixel_values=pixel_values)
                original_clip_embedding = F.normalize(original_clip_embedding, p=2, dim=1)

            # 3. 워터마크 생성 (CLIP → VAE → Watermark)
            watermark, original_latent = latent_watermarker.embed_watermark(original_clip_embedding)

            # 4. 개별 이미지의 latent 통계 계산 (mean, std)
            latent_mean = original_latent.cpu().numpy().flatten()  # [100]
            latent_std = np.abs(latent_mean) * 0.1  # 임시 std (실제로는 분포에서 계산)

            # 5. VINE 입력용 이미지 변환
            image_256 = to256(image).unsqueeze(0).to(device)   # [1,3,256,256]

            # 6. watermark_input shape 수정 → (1,100)
            watermark_input = watermark.to(device).float()     # [1,100]

            # 7. VINE 인코더 호출
            with torch.no_grad():
                encoded_256 = watermark_encoder(image_256, secret=watermark_input)  # [1,3,256,256]

            # 8. [-1,1] → [0,1] 변환 후 저장
            encoded_256 = (encoded_256 + 1.0) / 2.0     # [-1,1] → [0,1]
            encoded_256 = torch.clamp(encoded_256, 0, 1)

            watermarked_pil = transforms.ToPILImage()(encoded_256.squeeze(0).cpu())

            # 9. 저장
            output_path = os.path.join(output_dir, filename)
            watermarked_pil.save(output_path, quality=95)

            results.append({
                'filename': filename,
                'input_path': input_path,
                'output_path': output_path,
                'watermark': watermark.cpu().numpy(),
                'original_latent': original_latent.cpu().numpy(),
                'original_clip_embedding': original_clip_embedding.cpu().numpy(),  # 원본 CLIP 저장
                'latent_mean': latent_mean,  # 개별 latent 평균
                'latent_std': latent_std,    # 개별 latent 표준편차
                'status': 'success'
            })

        except Exception as e:
            print(f"\n❌ 오류 발생 ({filename}): {str(e)}")
            results.append({
                'filename': filename,
                'status': 'failed',
                'error': str(e)
            })

    # 통계 출력
    success_count = sum(1 for r in results if r['status'] == 'success')
    fail_count = len(results) - success_count

    print("="*80)
    print("워터마크 삽입 완료")
    print("="*80)
    print(f"  ✓ 성공: {success_count}장")
    if fail_count > 0:
        print(f"  ✗ 실패: {fail_count}장")
    print(f"  → 저장 경로: {output_dir}")
    print(f"  → 원본 CLIP embedding과 latent 통계 저장됨")
    print("="*80)

    return results

print("✅ 수정된 process_watermark_insertion 함수 준비 완료!")

In [ ]:
# 워터마크 삽입 실행
import pickle

print("="*80)
print("워터마크 삽입 시작")
print("="*80)
print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print("="*80)

insertion_results = process_watermark_insertion(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    clip_model=clip_model,
    clip_processor=clip_processor,
    watermark_encoder=watermark_encoder,
    latent_watermarker=latent_watermarker,
    device=device
)

# 결과를 pickle로 저장
results_path = os.path.join(DATASET_DIR, 'insertion_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(insertion_results, f)

print(f"\\n✅ 삽입 결과 저장: {results_path}")

In [ ]:
# 샘플 이미지 비교 (원본 vs 워터마크 삽입) — 모든 성공 케이스
success_cases = [r for r in insertion_results if r['status'] == 'success']

if len(success_cases) == 0:
    print("⚠️ 워터마크 삽입에 성공한 이미지가 없습니다.")
else:
    print("="*80)
    print(f"모든 성공 케이스 시각화 (총 {len(success_cases)}개)")
    print("="*80)

    for idx, sample in enumerate(success_cases):
        print(f"\n[{idx+1}/{len(success_cases)}] 파일명: {sample['filename']}")
        print("="*80)

        # 이미지 로드
        original_img = Image.open(sample['input_path'])
        watermarked_img = Image.open(sample['output_path'])

        # 시각화
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        axes[0].imshow(original_img)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(watermarked_img)
        axes[1].set_title('Watermarked Image (VINE)', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        plt.suptitle(f"Sample: {sample['filename']}", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

        # 워터마크 정보 출력
        watermark_bits = sample['watermark'].flatten()
        print(f"[Watermark Info]")
        print(f"  • Filename: {sample['filename']}")
        print(f"  • Watermark bits: {len(watermark_bits)} bits")
        print(f"  • Bit distribution: 1s={int(watermark_bits.sum())}, 0s={len(watermark_bits)-int(watermark_bits.sum())}")
        print(f"  • First 20 bits: {watermark_bits[:20].astype(int).tolist()}")
        print("="*80)

        # 너무 많은 figure 누적 방지
        plt.close(fig)

## 6. 워터마크 추출 및 의미 복원 (Edited → Analysis)

In [ ]:
def process_watermark_extraction(edited_dir, insertion_results, clip_model, clip_processor,
                                  vine_decoder, latent_watermarker, device='cuda'):
    """
    Process all edited images and extract watermarks, then reconstruct semantic embeddings
    개별 이미지의 원본 latent 통계를 사용하여 복원합니다.

    Pipeline:
        1. Load edited image from edited_dir
        2. Extract edited CLIP embedding (편집 후)
        3. Extract watermark using VINE Decoder (100-bit)
        4. Reconstruct CLIP embedding via VAE Decoder (워터마크 복원)
        5. Compare: Original CLIP vs Watermark-Reconstructed CLIP vs Edited CLIP

    Args:
        edited_dir: Edited image directory
        insertion_results: 삽입 결과 (원본 CLIP + latent 통계 포함)
        clip_model: CLIP model for embedding extraction
        clip_processor: CLIP processor
        vine_decoder: VINE decoder for watermark extraction
        latent_watermarker: LatentWatermarker instance
        device: cuda or cpu

    Returns:
        results: List of dicts with extraction and reconstruction information
    """

    # 파일명 → 삽입 정보 매핑 (확장자 무시)
    filename_to_insertion = {}
    basename_to_insertion = {}  # 확장자 제외한 파일명으로도 매핑

    if insertion_results:
        for item in insertion_results:
            if item['status'] == 'success':
                filename = item['filename']
                filename_to_insertion[filename] = item
                # 확장자 제외한 basename으로도 매핑
                basename = os.path.splitext(filename)[0]
                basename_to_insertion[basename] = item

    # 이미지 파일 목록
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    image_files = [f for f in os.listdir(edited_dir)
                   if f.lower().endswith(image_extensions)]

    if len(image_files) == 0:
        print(f"❌ {edited_dir}에서 이미지를 찾을 수 없습니다.")
        return []

    print(f"\\n발견된 편집 이미지: {len(image_files)}장")
    print("="*80)

    results = []

    for idx, filename in enumerate(tqdm(image_files, desc="워터마크 추출 중")):
        try:
            # 1. 편집된 이미지 로드
            edited_path = os.path.join(edited_dir, filename)
            edited_image = Image.open(edited_path).convert('RGB')

            # 2. 삽입 정보 가져오기 (확장자 무시 매칭)
            insertion_info = filename_to_insertion.get(filename, None)

            # 정확한 파일명으로 못 찾으면 basename으로 시도
            if insertion_info is None:
                basename = os.path.splitext(filename)[0]
                insertion_info = basename_to_insertion.get(basename, None)

            if insertion_info is None:
                print(f"\\n⚠️ {filename}: 삽입 정보를 찾을 수 없습니다. 건너뜁니다.")
                print(f"   (시도한 파일명: {filename}, basename: {os.path.splitext(filename)[0]})")
                continue

            # 3. 원본 CLIP embedding 가져오기
            original_clip_embedding = insertion_info['original_clip_embedding']
            original_clip_tensor = torch.from_numpy(original_clip_embedding).to(device)

            # 4. 개별 latent 통계 가져오기
            latent_stats = (insertion_info['latent_mean'], insertion_info['latent_std'])

            # 5. 편집된 이미지의 CLIP 임베딩 추출
            inputs = clip_processor(images=edited_image, return_tensors="pt", padding=True)
            pixel_values = inputs['pixel_values'].to(device)

            with torch.no_grad():
                edited_clip_embedding = clip_model.get_image_features(pixel_values=pixel_values)
                edited_clip_embedding = F.normalize(edited_clip_embedding, p=2, dim=1)

            # 6. VINE Decoder로 워터마크 추출
            # VINE은 [-1, 1] 범위를 기대함
            edited_transform = transforms.Compose([
                transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Lambda(lambda x: x * 2.0 - 1.0),  # [0,1] → [-1,1]
            ])
            edited_tensor = edited_transform(edited_image).unsqueeze(0).to(device)

            with torch.no_grad():
                extracted_watermark_raw = vine_decoder(edited_tensor)

                # 디버깅: 원시 출력값 확인 (첫 번째 이미지만)
                if idx == 0:
                    print(f"\n[Debug] VINE Decoder raw output:")
                    print(f"  Shape: {extracted_watermark_raw.shape}")
                    print(f"  Min: {extracted_watermark_raw.min():.4f}")
                    print(f"  Max: {extracted_watermark_raw.max():.4f}")
                    print(f"  Mean: {extracted_watermark_raw.mean():.4f}")
                    print(f"  Std: {extracted_watermark_raw.std():.4f}")
                    print(f"  First 10 values: {extracted_watermark_raw[0, :10].cpu().numpy()}")

                # demo_war.ipynb 방식: numpy로 변환 후 round
                decoded_watermark_np = np.array(extracted_watermark_raw[0].cpu().detach())
                decoded_watermark_np = np.round(decoded_watermark_np).astype(int)

                # 다시 텐서로 변환
                extracted_watermark = torch.from_numpy(decoded_watermark_np).float().unsqueeze(0).to(device)

            # 7. 워터마크 → VAE Decoder → CLIP 임베딩 복원 (개별 통계 사용)
            watermark_reconstructed_clip, reconstructed_latent = latent_watermarker.extract_and_reconstruct(
                extracted_watermark, latent_stats=latent_stats
            )

            # 8. 거리 계산
            # 8.1 원본 CLIP vs 워터마크 복원 CLIP
            cosine_original_vs_watermark = F.cosine_similarity(
                original_clip_tensor, watermark_reconstructed_clip, dim=1
            ).item()
            euclidean_original_vs_watermark = torch.norm(
                original_clip_tensor - watermark_reconstructed_clip, p=2, dim=1
            ).item()

            # 8.2 원본 CLIP vs 편집된 CLIP
            cosine_original_vs_edited = F.cosine_similarity(
                original_clip_tensor, edited_clip_embedding, dim=1
            ).item()
            euclidean_original_vs_edited = torch.norm(
                original_clip_tensor - edited_clip_embedding, p=2, dim=1
            ).item()

            # 8.3 워터마크 복원 CLIP vs 편집된 CLIP
            cosine_watermark_vs_edited = F.cosine_similarity(
                watermark_reconstructed_clip, edited_clip_embedding, dim=1
            ).item()
            euclidean_watermark_vs_edited = torch.norm(
                watermark_reconstructed_clip - edited_clip_embedding, p=2, dim=1
            ).item()

            # 결과 저장
            results.append({
                'filename': filename,
                'edited_path': edited_path,
                'original_clip_embedding': original_clip_embedding,
                'watermark_reconstructed_clip_embedding': watermark_reconstructed_clip.cpu().numpy(),
                'edited_clip_embedding': edited_clip_embedding.cpu().numpy(),
                'extracted_watermark': extracted_watermark.cpu().numpy(),
                'reconstructed_latent': reconstructed_latent.cpu().numpy(),
                'latent_stats': latent_stats,
                # 거리 메트릭
                'cosine_original_vs_watermark': cosine_original_vs_watermark,
                'euclidean_original_vs_watermark': euclidean_original_vs_watermark,
                'cosine_original_vs_edited': cosine_original_vs_edited,
                'euclidean_original_vs_edited': euclidean_original_vs_edited,
                'cosine_watermark_vs_edited': cosine_watermark_vs_edited,
                'euclidean_watermark_vs_edited': euclidean_watermark_vs_edited,
                'status': 'success'
            })

        except Exception as e:
            print(f"\\n❌ 오류 발생 ({filename}): {str(e)}")
            results.append({
                'filename': filename,
                'status': 'failed',
                'error': str(e)
            })

    # 통계 출력
    success_count = sum(1 for r in results if r['status'] == 'success')
    fail_count = len(results) - success_count

    if success_count > 0:
        success_results = [r for r in results if r['status'] == 'success']

        avg_cos_orig_vs_wm = np.mean([r['cosine_original_vs_watermark'] for r in success_results])
        avg_cos_orig_vs_edit = np.mean([r['cosine_original_vs_edited'] for r in success_results])
        avg_cos_wm_vs_edit = np.mean([r['cosine_watermark_vs_edited'] for r in success_results])

        print("\\n" + "="*80)
        print("워터마크 추출 및 의미 복원 완료")
        print("="*80)
        print(f"  ✓ 성공: {success_count}장")
        if fail_count > 0:
            print(f"  ✗ 실패: {fail_count}장")

        print(f"\\n[CLIP Embedding 거리 분석]")
        print(f"  1️⃣ 원본 vs 워터마크 복원:")
        print(f"     • Cosine Similarity: {avg_cos_orig_vs_wm:.4f}")
        print(f"  2️⃣ 원본 vs 편집된 이미지:")
        print(f"     • Cosine Similarity: {avg_cos_orig_vs_edit:.4f}")
        print(f"  3️⃣ 워터마크 복원 vs 편집된 이미지:")
        print(f"     • Cosine Similarity: {avg_cos_wm_vs_edit:.4f}")
        print("="*80)
    else:
        print("\\n⚠️ 성공한 추출이 없습니다.")

    return results

print("✅ 워터마크 추출 함수 정의 완료 (통계 기반 복원)")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

def show_images_in_directory(directory, columns=3, max_images=30):
    """
    지정한 디렉토리 안의 이미지를 grid 형태로 시각화.
    max_images: 너무 많을 때 최대 몇 개까지 보여줄지 설정.
    """
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    files = [f for f in os.listdir(directory)
             if f.lower().endswith(image_extensions)]

    if len(files) == 0:
        print(f"❌ '{directory}' 에서 표시할 이미지가 없습니다.")
        return

    print(f"✓ 총 {len(files)}장의 이미지 발견")

    # 너무 많으면 잘라줌
    files = files[:max_images]

    rows = (len(files) + columns - 1) // columns

    plt.figure(figsize=(5 * columns, 5 * rows))

    for i, filename in enumerate(files):
        img_path = os.path.join(directory, filename)
        try:
            img = Image.open(img_path).convert("RGB")

            plt.subplot(rows, columns, i + 1)
            plt.imshow(img)
            plt.title(filename)
            plt.axis("off")
        except:
            print(f"⚠️ 이미지 로드 실패: {filename}")

    plt.tight_layout()
    plt.show()


# 실행
print("📸 편집된 이미지 시각화")
show_images_in_directory(EDITED_DIR, columns=3)

In [ ]:
# 워터마크 추출 실행
print("="*80)
print("워터마크 추출 및 의미 복원 시작 (통계 기반 복원)")
print("="*80)
print(f"Edited images: {EDITED_DIR}")
print("="*80)

extraction_results = process_watermark_extraction(
    edited_dir=EDITED_DIR,
    insertion_results=insertion_results,  # 원본 CLIP + latent 통계 전달
    clip_model=clip_model,
    clip_processor=clip_processor,
    vine_decoder=vine_decoder,
    latent_watermarker=latent_watermarker,
    device=device
)

# 결과 저장
results_path = os.path.join(DATASET_DIR, 'extraction_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(extraction_results, f)

print(f"✅ 추출 결과 저장: {results_path}")

### 🔍 워터마크 삽입/추출 검증 테스트

편집 전에 워터마크가 제대로 삽입되었는지 확인합니다.

In [ ]:
# 전체 이미지 워터마크 비트 정확도 분석
if len(extraction_results) > 0:
    print("="*80)
    print("전체 이미지 워터마크 비트 정확도 분석")
    print("="*80)

    # insertion_results를 파일명(basename)으로 매핑
    insertion_map = {}
    for item in insertion_results:
        if item['status'] == 'success':
            basename = os.path.splitext(item['filename'])[0]
            insertion_map[basename] = item

    # 전체 이미지에 대해 분석
    total_images = 0
    successful_extractions = 0
    all_bit_accuracies = []
    all_cosine_orig_wm = []
    all_cosine_orig_edit = []
    all_cosine_wm_edit = []

    print(f"\n📊 이미지별 워터마크 비트 정확도 (100비트 전체 비교):\n")
    print(f"{'번호':<6} {'파일명':<25} {'Bit 정확도':<15} {'일치/전체':<12} {'1s 분포':<12} {'거리 메트릭 (Orig vs WM)':<20}")
    print("-" * 110)

    for idx, result in enumerate(extraction_results, 1):
        if result['status'] != 'success':
            continue

        total_images += 1
        filename = result['filename']
        basename = os.path.splitext(filename)[0]

        # 원본 워터마크 가져오기
        if basename not in insertion_map:
            print(f"{idx:<6} {filename:<25} ⚠️ 원본 워터마크를 찾을 수 없음")
            continue

        original_watermark = insertion_map[basename]['watermark'].flatten()
        extracted_watermark = result['extracted_watermark'].flatten()

        # 100비트 전체 비교
        same_bits = sum(o == e for o, e in zip(original_watermark, extracted_watermark))
        bit_accuracy = same_bits / 100

        # 추출된 워터마크의 1s 비율
        ones_ratio = extracted_watermark.sum() / 100

        # 거리 메트릭
        cosine_orig_wm = result['cosine_original_vs_watermark']
        cosine_orig_edit = result['cosine_original_vs_edited']
        cosine_wm_edit = result['cosine_watermark_vs_edited']

        # 저장
        all_bit_accuracies.append(bit_accuracy)
        all_cosine_orig_wm.append(cosine_orig_wm)
        all_cosine_orig_edit.append(cosine_orig_edit)
        all_cosine_wm_edit.append(cosine_wm_edit)
        successful_extractions += 1

        # 출력
        status_icon = "✅" if bit_accuracy >= 0.85 else "⚠️" if bit_accuracy >= 0.70 else "❌"
        print(f"{idx:<6} {filename:<25} {status_icon} {bit_accuracy*100:>5.1f}%       {same_bits:>3}/100     1s={extracted_watermark.sum():>3}/100   Cos:{cosine_orig_wm:.4f}")

    print("-" * 110)

    # 통계 요약
    if len(all_bit_accuracies) > 0:
        avg_accuracy = np.mean(all_bit_accuracies) * 100
        min_accuracy = np.min(all_bit_accuracies) * 100
        max_accuracy = np.max(all_bit_accuracies) * 100
        std_accuracy = np.std(all_bit_accuracies) * 100

        # 정확도 분포
        excellent = sum(1 for acc in all_bit_accuracies if acc >= 0.85)
        good = sum(1 for acc in all_bit_accuracies if 0.70 <= acc < 0.85)
        poor = sum(1 for acc in all_bit_accuracies if acc < 0.70)

        print(f"\n📈 전체 통계 ({successful_extractions}장):")
        print(f"  • 평균 Bit 정확도: {avg_accuracy:.2f}%")
        print(f"  • 최소/최대: {min_accuracy:.2f}% / {max_accuracy:.2f}%")
        print(f"  • 표준편차: {std_accuracy:.2f}%")
        print(f"\n  • 정확도 분포:")
        print(f"    - Excellent (≥85%): {excellent}장 ({excellent/successful_extractions*100:.1f}%)")
        print(f"    - Good (70-85%): {good}장 ({good/successful_extractions*100:.1f}%)")
        print(f"    - Poor (<70%): {poor}장 ({poor/successful_extractions*100:.1f}%)")

        print(f"\n  • CLIP Cosine Similarity (평균):")
        print(f"    - 원본 vs 워터마크 복원: {np.mean(all_cosine_orig_wm):.4f}")
        print(f"    - 원본 vs 편집된 이미지: {np.mean(all_cosine_orig_edit):.4f}")
        print(f"    - 워터마크 복원 vs 편집된 이미지: {np.mean(all_cosine_wm_edit):.4f}")

    print("="*80)
else:
    print("\\n⚠️ 워터마크 추출 결과가 없습니다.")

In [ ]:
# 각 이미지별 100비트 워터마크 상세 비교
if len(extraction_results) > 0:
    # insertion_results를 파일명(basename)으로 매핑
    insertion_map = {}
    for item in insertion_results:
        if item['status'] == 'success':
            basename = os.path.splitext(item['filename'])[0]
            insertion_map[basename] = item

    print("="*100)
    print("개별 이미지 워터마크 비트 상세 비교 (100비트 전체)")
    print("="*100)

    for idx, result in enumerate(extraction_results, 1):
        if result['status'] != 'success':
            continue

        filename = result['filename']
        basename = os.path.splitext(filename)[0]

        # 원본 워터마크 가져오기
        if basename not in insertion_map:
            print(f"\n[{idx}] {filename}: ⚠️ 원본 워터마크를 찾을 수 없음")
            continue

        original_watermark = insertion_map[basename]['watermark'].flatten()
        extracted_watermark = result['extracted_watermark'].flatten()

        # 100비트 전체 비교
        same_bits = sum(o == e for o, e in zip(original_watermark, extracted_watermark))
        bit_accuracy = same_bits / 100

        print(f"\n{'='*100}")
        print(f"[{idx}] {filename}")
        print(f"{'='*100}")

        print(f"\n📊 Bit Accuracy: {bit_accuracy*100:.2f}% ({same_bits}/100 bits 일치)")

        # 비트 분포
        orig_ones = original_watermark.sum()
        extr_ones = extracted_watermark.sum()
        print(f"\n🔢 Bit 분포:")
        print(f"   Original:  1s={orig_ones}/100, 0s={100-orig_ones}/100")
        print(f"   Extracted: 1s={extr_ones}/100, 0s={100-extr_ones}/100")

        # 오류 비트 분석
        errors = [i for i, (o, e) in enumerate(zip(original_watermark, extracted_watermark)) if o != e]
        if len(errors) > 0:
            print(f"\n❌ 오류 비트: {len(errors)}개 위치")
            print(f"   오류 위치 (전체): {errors}")
        else:
            print(f"\n✅ 완벽한 일치!")

        # 100비트 전체 시각적 비교 (10비트씩 10줄)
        print(f"\n🔍 100비트 전체 비교 (Original vs Extracted):")
        print(f"{'':>10} " + "".join([f"{i:>10}" for i in range(0, 100, 10)]))
        print(f"{'':>10} " + "".join([f"{'Bits ' + str(i) + '-' + str(i+9):>10}" for i in range(0, 100, 10)]))
        print("-" * 110)

        for row in range(10):
            start_idx = row * 10
            end_idx = start_idx + 10

            orig_bits = ''.join(str(int(b)) for b in original_watermark[start_idx:end_idx])
            extr_bits = ''.join(str(int(b)) for b in extracted_watermark[start_idx:end_idx])

            # 차이가 있는 비트 표시
            match_markers = ''.join(['✓' if o == e else '✗' for o, e in zip(original_watermark[start_idx:end_idx], extracted_watermark[start_idx:end_idx])])

            print(f"Bits {start_idx:>2}-{end_idx-1:>2}:")
            print(f"  Original:  {orig_bits}")
            print(f"  Extracted: {extr_bits}")
            print(f"  Match:     {match_markers}")

        # 거리 메트릭
        print(f"\n📏 CLIP 거리 메트릭:")
        print(f"   Original vs Watermark 복원:")
        print(f"      • Cosine Similarity: {result['cosine_original_vs_watermark']:.4f}")
        print(f"      • Euclidean Distance: {result['euclidean_original_vs_watermark']:.4f}")
        print(f"   Original vs Edited:")
        print(f"      • Cosine Similarity: {result['cosine_original_vs_edited']:.4f}")
        print(f"      • Euclidean Distance: {result['euclidean_original_vs_edited']:.4f}")
        print(f"   Watermark 복원 vs Edited:")
        print(f"      • Cosine Similarity: {result['cosine_watermark_vs_edited']:.4f}")
        print(f"      • Euclidean Distance: {result['euclidean_watermark_vs_edited']:.4f}")

    print(f"\n{'='*100}")
    print("✅ 전체 이미지 상세 비교 완료")
    print(f"{'='*100}")
else:
    print("\n⚠️ 워터마크 추출 결과가 없습니다.")

### 6.3 상세 워터마크 복원 및 CLIP 비교 분석 (단일 이미지 예제)

In [ ]:
# ============================================================================
# 전체 이미지: 편집된 이미지에서 워터마크 추출 및 CLIP 복원
# ============================================================================

print("\n" + "="*80)
print("전체 이미지 워터마크 복원 및 상세 분석")
print("="*80)

if len(extraction_results) > 0:
    # insertion_results를 딕셔너리로 변환 (빠른 조회)
    insertion_map = {}
    for item in insertion_results:
        if item['status'] == 'success':
            basename = os.path.splitext(item['filename'])[0]
            insertion_map[basename] = item

    # 전체 이미지에 대한 워터마크 정보 저장
    all_watermark_analysis = []

    print(f"\n총 {len(extraction_results)}개 이미지 처리 중...")

    for idx, sample_result in enumerate(extraction_results, 1):
        if sample_result['status'] != 'success':
            continue

        test_filename = sample_result['filename']
        basename = os.path.splitext(test_filename)[0]

        if basename not in insertion_map:
            print(f"⚠️ {test_filename}: 원본 정보를 찾을 수 없습니다.")
            continue

        insertion_info = insertion_map[basename]

        # 이미 extraction_results에 워터마크가 있으므로 재사용
        decoded_watermark = sample_result['extracted_watermark'].flatten()
        test_watermark = insertion_info['watermark'].flatten()

        # 워터마크 비교
        same_bits = sum(o == e for o, e in zip(test_watermark, decoded_watermark))
        bit_accuracy = same_bits / 100

        # 저장
        all_watermark_analysis.append({
            'filename': test_filename,
            'original_watermark': test_watermark,
            'decoded_watermark': decoded_watermark,
            'bit_accuracy': bit_accuracy,
            'same_bits': same_bits,
            'insertion_info': insertion_info
        })

    print(f"\n✅ {len(all_watermark_analysis)}개 이미지 워터마크 분석 완료")
    print("="*80)

else:
    print("\n⚠️ 추출 결과가 없습니다. 워터마크 추출을 먼저 실행하세요.")
    all_watermark_analysis = []

### Training 데이터 clip embedding & latent 통계 계산

In [ ]:
def load_images_from_category(base_path, category, max_images=None):
    """카테고리별 이미지 경로 로드"""
    category_path = os.path.join(base_path, category)
    image_files = []

    for file in os.listdir(category_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            image_files.append(os.path.join(category_path, file))

    if max_images:
        image_files = image_files[:max_images]

    return image_files

def extract_clip_embeddings(image_paths, model, processor, device, batch_size=32):
    """CLIP 이미지 임베딩 추출"""
    embeddings = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting CLIP embeddings"):
            batch_paths = image_paths[i:i+batch_size]
            batch_images = []

            for img_path in batch_paths:
                try:
                    image = Image.open(img_path).convert('RGB')
                    batch_images.append(image)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
                    continue

            if batch_images:
                inputs = processor(images=batch_images, return_tensors="pt", padding=True)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                image_features = model.get_image_features(**inputs)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                embeddings.append(image_features.cpu().numpy())

    return np.vstack(embeddings)

print("✅ 데이터 로드 함수 정의 완료")

In [ ]:
# semantic_wm 데이터셋 로드
train_path = TRAINING_DATA_DIR
categories = ['normal', 'violence', 'sexual']

all_embeddings = []
all_labels = []
category_stats = {}

# 카테고리당 최대 이미지 수 설정 (메모리 절약)
max_images_per_category = None

print("\n" + "="*80)
print("Training Data 로드 중...")
print("="*80)

for category in categories:
    print(f"\n[{category.upper()}] 카테고리 처리 중...")

    # 이미지 경로 로드
    image_paths = load_images_from_category(train_path, category, max_images_per_category)
    print(f"  - 로드된 이미지 수: {len(image_paths)}")

    # CLIP 임베딩 추출
    embeddings = extract_clip_embeddings(image_paths, clip_model, clip_processor, device)

    all_embeddings.append(embeddings)
    all_labels.extend([category] * len(embeddings))
    category_stats[category] = len(embeddings)

    print(f"  - 추출된 임베딩 shape: {embeddings.shape}")
    print(f"  ✓ {category} 완료")

# 전체 데이터 결합
all_embeddings = np.vstack(all_embeddings)
all_labels = np.array(all_labels)

print("\n" + "="*80)
print("Training Data 로드 완료")
print("="*80)
print(f"전체 임베딩 shape: {all_embeddings.shape}")
print(f"전체 레이블 수: {len(all_labels)}")
print("\n카테고리별 통계:")
for cat, count in category_stats.items():
    print(f"  - {cat.capitalize()}: {count} images ({count/len(all_labels)*100:.1f}%)")
print("="*80)

In [ ]:
# CLIP Embedding Space t-SNE 계산
print("\nCLIP Embedding Space t-SNE 계산 중...")
clip_tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
clip_embeddings_2d = clip_tsne.fit_transform(all_embeddings)

print(f"✅ CLIP t-SNE 완료: {clip_embeddings_2d.shape}")
print(f"   - 원본 차원: {all_embeddings.shape[1]}D (CLIP)")
print(f"   - 축소 차원: 2D (t-SNE)")

In [ ]:
# CLIP Embedding Space 시각화
plt.figure(figsize=(14, 10))

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for category in categories:
    mask = all_labels == category
    plt.scatter(
        clip_embeddings_2d[mask, 0],
        clip_embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=f'{category.capitalize()}',
        alpha=0.6,
        s=50,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('CLIP Embedding Space t-SNE Visualization (512D → 2D)',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=12, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('/content/clip_embedding_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ CLIP Embedding t-SNE 시각화 저장: /content/clip_embedding_tsne.png")

In [ ]:
# Latent Space 추출
print("Latent Space 추출 중...")
vae_model.eval()
with torch.no_grad():
    all_embeddings_tensor = torch.FloatTensor(all_embeddings).to(device)
    all_latent_mu, _ = vae_model.encode(all_embeddings_tensor)
    all_latent_vectors = all_latent_mu.cpu().numpy()

print(f"✅ Latent vectors shape: {all_latent_vectors.shape}")
print(f"   - Mean: {all_latent_vectors.mean():.4f}")
print(f"   - Std: {all_latent_vectors.std():.4f}")
print(f"   - Min: {all_latent_vectors.min():.4f}")
print(f"   - Max: {all_latent_vectors.max():.4f}")

In [ ]:
# Latent Space t-SNE
print("Latent Space t-SNE 계산 중...")
latent_tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
latent_embeddings_2d = latent_tsne.fit_transform(all_latent_vectors)

print(f"✅ Latent t-SNE 완료: {latent_embeddings_2d.shape}")

In [ ]:
# Latent Space 시각화
plt.figure(figsize=(14, 10))

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

for category in categories:
    mask = all_labels == category
    plt.scatter(
        latent_embeddings_2d[mask, 0],
        latent_embeddings_2d[mask, 1],
        c=colors[category],
        marker=markers[category],
        label=f'{category.capitalize()}',
        alpha=0.6,
        s=50,
        edgecolors='white',
        linewidth=0.5
    )

plt.title('Latent Space t-SNE Visualization (100D → 2D)',
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(fontsize=12, loc='best', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('/content/latent_space_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Latent Space t-SNE 시각화 저장: /content/latent_space_tsne.png")

In [ ]:
# ============================================================================
# 전체 이미지: CLIP Embedding 추출 및 복원 (Training 통계 사용)
# ============================================================================

if len(all_watermark_analysis) > 0:

    print("\n" + "="*80)
    print("전체 이미지 CLIP Embedding 복원 중...")
    print("="*80)

    # Training data 전체 통계 확인
    if 'all_latent_vectors' in locals() and all_latent_vectors is not None:
        global_latent_stats = (all_latent_vectors.mean(axis=0), all_latent_vectors.std(axis=0))
        use_global_stats = True
        print(f"\n✅ Training 통계 사용: shape={all_latent_vectors.shape}")
    else:
        print("\n⚠️ all_latent_vectors가 없습니다. 개별 이미지 통계를 사용합니다.")
        use_global_stats = False

    # 전체 이미지에 대한 CLIP 복원
    all_clip_analysis = []

    vae_model.eval()
    clip_model.eval()

    for idx, wm_info in enumerate(all_watermark_analysis, 1):
        try:
            test_filename = wm_info['filename']
            insertion_info = wm_info['insertion_info']

            # 통계 선택
            if use_global_stats:
                latent_stats = global_latent_stats
            else:
                latent_stats = (insertion_info['latent_mean'], insertion_info['latent_std'])

            # 1. 원본 워터마크 → CLIP 복원
            test_watermark = wm_info['original_watermark']
            test_watermark_tensor = torch.from_numpy(test_watermark).float().unsqueeze(0).to(device)
            test_latent_restored = latent_watermarker.watermark_to_latent(test_watermark_tensor, latent_stats)

            with torch.no_grad():
                test_clip_reconstructed = vae_model.decode(test_latent_restored)
                test_clip_reconstructed_np = test_clip_reconstructed.cpu().numpy()[0]

            # 2. 편집된 워터마크 → CLIP 복원
            decoded_watermark = wm_info['decoded_watermark']
            decoded_watermark_tensor = torch.from_numpy(decoded_watermark).float().unsqueeze(0).to(device)
            decoded_latent_restored = latent_watermarker.watermark_to_latent(decoded_watermark_tensor, latent_stats)

            with torch.no_grad():
                decoded_clip_reconstructed = vae_model.decode(decoded_latent_restored)
                decoded_clip_reconstructed_np = decoded_clip_reconstructed.cpu().numpy()[0]

            # 3. 편집된 이미지의 CLIP embedding (이미 extraction_results에 있음)
            edited_clip_embedding_np = None
            for result in extraction_results:
                if result['filename'] == test_filename:
                    edited_clip_embedding_np = result['edited_clip_embedding'].flatten()
                    break

            # 4. 원본 CLIP
            test_clip_embedding_np = insertion_info['original_clip_embedding'].flatten()

            # 저장
            all_clip_analysis.append({
                'filename': test_filename,
                'original_clip': test_clip_embedding_np,
                'original_wm_reconstructed_clip': test_clip_reconstructed_np,
                'edited_clip': edited_clip_embedding_np,
                'edited_wm_reconstructed_clip': decoded_clip_reconstructed_np,
                'bit_accuracy': wm_info['bit_accuracy']
            })

        except Exception as e:
            print(f"⚠️ {test_filename}: CLIP 복원 실패 - {str(e)}")
            continue

    print(f"\n✅ {len(all_clip_analysis)}개 이미지 CLIP Embedding 복원 완료")
    print("="*80)

else:
    print("\n⚠️ 워터마크 분석 결과가 없습니다.")
    all_clip_analysis = []

In [ ]:
# ============================================================================
# 전체 이미지: CLIP 유사도 계산 (6가지 메트릭)
# ============================================================================

if len(all_clip_analysis) > 0:

    print("\n" + "="*80)
    print("전체 이미지 CLIP EMBEDDING SIMILARITY ANALYSIS")
    print("="*80)

    # 각 이미지별로 6가지 유사도 계산
    all_similarities = []

    for clip_info in all_clip_analysis:
        test_clip_embedding_np = clip_info['original_clip']
        test_clip_reconstructed_np = clip_info['original_wm_reconstructed_clip']
        edited_clip_embedding_np = clip_info['edited_clip']
        decoded_clip_reconstructed_np = clip_info['edited_wm_reconstructed_clip']

        # 6가지 유사도 계산
        sim_orig_vs_edit = np.dot(test_clip_embedding_np, edited_clip_embedding_np)
        sim_orig_vs_orig_recon = np.dot(test_clip_embedding_np, test_clip_reconstructed_np)
        sim_orig_vs_edit_recon = np.dot(test_clip_embedding_np, decoded_clip_reconstructed_np)
        sim_edit_vs_orig_recon = np.dot(edited_clip_embedding_np, test_clip_reconstructed_np)
        sim_edit_vs_edit_recon = np.dot(edited_clip_embedding_np, decoded_clip_reconstructed_np)
        sim_orig_recon_vs_edit_recon = np.dot(test_clip_reconstructed_np, decoded_clip_reconstructed_np)

        all_similarities.append({
            'filename': clip_info['filename'],
            'bit_accuracy': clip_info['bit_accuracy'],
            'sim_orig_vs_edit': sim_orig_vs_edit,
            'sim_orig_vs_orig_recon': sim_orig_vs_orig_recon,
            'sim_orig_vs_edit_recon': sim_orig_vs_edit_recon,
            'sim_edit_vs_orig_recon': sim_edit_vs_orig_recon,
            'sim_edit_vs_edit_recon': sim_edit_vs_edit_recon,
            'sim_orig_recon_vs_edit_recon': sim_orig_recon_vs_edit_recon
        })

    # 통계 계산
    avg_sim = {
        'orig_vs_edit': np.mean([s['sim_orig_vs_edit'] for s in all_similarities]),
        'orig_vs_orig_recon': np.mean([s['sim_orig_vs_orig_recon'] for s in all_similarities]),
        'orig_vs_edit_recon': np.mean([s['sim_orig_vs_edit_recon'] for s in all_similarities]),
        'edit_vs_orig_recon': np.mean([s['sim_edit_vs_orig_recon'] for s in all_similarities]),
        'edit_vs_edit_recon': np.mean([s['sim_edit_vs_edit_recon'] for s in all_similarities]),
        'orig_recon_vs_edit_recon': np.mean([s['sim_orig_recon_vs_edit_recon'] for s in all_similarities])
    }

    avg_bit_accuracy = np.mean([s['bit_accuracy'] for s in all_similarities])

    # 결과 출력
    print(f"\n📊 전체 {len(all_similarities)}개 이미지 평균 유사도:")
    print(f"\n[1] 원본 vs 편집/복원")
    print(f"  Original vs Edited:                            {avg_sim['orig_vs_edit']:.4f}")
    print(f"  Original vs Orig WM Reconstructed:             {avg_sim['orig_vs_orig_recon']:.4f}")
    print(f"  Original vs Edited WM Reconstructed:           {avg_sim['orig_vs_edit_recon']:.4f}")

    print(f"\n[2] 편집된 이미지 vs 복원")
    print(f"  Edited vs Orig WM Reconstructed:               {avg_sim['edit_vs_orig_recon']:.4f}")
    print(f"  Edited vs Edited WM Reconstructed:             {avg_sim['edit_vs_edit_recon']:.4f}")

    print(f"\n[3] 두 복원 CLIP 비교")
    print(f"  Orig WM Recon vs Edited WM Recon:              {avg_sim['orig_recon_vs_edit_recon']:.4f}")

    print(f"\n[Interpretation]")
    print(f"  - 평균 Edit 변형 정도: {1 - avg_sim['orig_vs_edit']:.4f}")
    print(f"  - 평균 워터마크 bit accuracy: {avg_bit_accuracy:.1%}")
    print(f"  - 평균 원본 워터마크 → CLIP 복원 정확도: {avg_sim['orig_vs_orig_recon']:.4f}")
    print(f"  - 평균 편집 워터마크 → CLIP 복원 정확도: {avg_sim['edit_vs_edit_recon']:.4f}")
    print(f"  - 평균 두 복원 CLIP 간 유사도: {avg_sim['orig_recon_vs_edit_recon']:.4f}")
    print(f"  ✅ Watermark를 통해 원본 의미 보존 가능!")
    print("="*80)

    # 개별 이미지 결과 저장 (필요시 사용)
    # all_similarities 변수에 모든 이미지의 유사도가 저장되어 있음

else:
    print("\n⚠️ CLIP 유사도 계산을 위한 데이터가 없습니다.")
    all_similarities = []

## 7. CLIP Embedding 거리 분석 및 시각화

In [ ]:
# 7.1 이미지별 거리 메트릭 비교
if len(extraction_results) > 0:
    print("="*80)
    print("7.1 이미지별 CLIP Embedding 거리 분석")
    print("="*80)

    success_results = [r for r in extraction_results if r['status'] == 'success']

    if len(success_results) > 0:
        # 데이터 준비
        filenames = [r['filename'] for r in success_results]
        cos_orig_wm = [r['cosine_original_vs_watermark'] for r in success_results]
        cos_orig_edit = [r['cosine_original_vs_edited'] for r in success_results]
        cos_wm_edit = [r['cosine_watermark_vs_edited'] for r in success_results]

        # 바차트
        fig, ax = plt.subplots(figsize=(16, 6))

        x = np.arange(len(filenames))
        width = 0.25

        bars1 = ax.bar(x - width, cos_orig_wm, width, label='Original vs Watermark', alpha=0.8, color='green')
        bars2 = ax.bar(x, cos_orig_edit, width, label='Original vs Edited', alpha=0.8, color='red')
        bars3 = ax.bar(x + width, cos_wm_edit, width, label='Watermark vs Edited', alpha=0.8, color='orange')

        ax.set_xlabel('Image Index', fontsize=14, fontweight='bold')
        ax.set_ylabel('Cosine Similarity', fontsize=14, fontweight='bold')
        ax.set_title('CLIP Embedding Distances per Image', fontsize=16, fontweight='bold', pad=20)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{i+1}" for i in range(len(filenames))], rotation=0)
        ax.legend(fontsize=12)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_ylim([0, 1.0])

        plt.tight_layout()
        plt.show()

        print(f"✅ {len(success_results)}개 이미지 거리 분석 완료")
else:
    print("⚠️ 추출 결과가 없습니다.")

## 8. 종합 t-SNE 시각화: Training + 원본 + 워터마크 복원 + 편집된 CLIP

In [ ]:
from sklearn.manifold import TSNE

print("\n" + "="*80)
print("📌 개별 이미지 t-SNE 시각화 생성 중...")
print("="*80)

# 카테고리 색상
# color_map = {
#     'normal':   '#2ecc71',
#     'violence': '#e74c3c',
#     'sexual':   '#9b59b6'
# }

colors = {'normal': '#2ecc71', 'violence': '#e74c3c', 'sexual': '#9b59b6'}
markers = {'normal': 'o', 'violence': '^', 'sexual': 's'}

training_labels_arr = np.array(all_labels)
num_train = len(training_labels_arr)

for item in all_clip_analysis:
    fname = item['filename']
    print(f"\n→ {fname} 처리 중...")

    # test embeddings
    e_original = item['original_clip'].reshape(1, -1)
    e_original_rec = item['original_wm_reconstructed_clip'].reshape(1, -1)
    e_edited = item['edited_clip'].reshape(1, -1)
    e_edited_rec = item['edited_wm_reconstructed_clip'].reshape(1, -1)

    # t-SNE 입력
    combined = np.vstack([
        all_embeddings,
        e_original,
        e_original_rec,
        e_edited,
        e_edited_rec
    ])

    tsne = TSNE(n_components=2, n_iter=1000, perplexity=30, random_state=42)
    tsne_2d = tsne.fit_transform(combined)

    # 인덱스 정리
    idx_train = tsne_2d[:num_train]
    idx_o  = tsne_2d[num_train + 0]
    idx_or = tsne_2d[num_train + 1]
    idx_e  = tsne_2d[num_train + 2]
    idx_er = tsne_2d[num_train + 3]

    # 그리기
    plt.figure(figsize=(16, 12))

    # 1) Training 데이터 배경
    for category in ['normal', 'violence', 'sexual']:
      mask = (all_labels == category)
      plt.scatter(
          clip_embeddings_2d[mask, 0],                 # t-SNE x
          clip_embeddings_2d[mask, 1],                 # t-SNE y
          c=colors[category],
          marker=markers[category],
          label=f'{category.capitalize()} (training)',
          alpha=0.35,
          s=25,
          edgecolors='white',
          linewidth=0.3
      )

    # 2) 각 단계 점 찍기
    plt.scatter(idx_o[0],  idx_o[1],  c='blue',   s=500, marker='*',
                edgecolor='black', linewidth=2, label="1. Original CLIP")

    plt.scatter(idx_or[0], idx_or[1], c='cyan',   s=500, marker='*',
                edgecolor='black', linewidth=2, label="2. Reconstructed (original WM)")

    plt.scatter(idx_e[0],  idx_e[1],  c='red',    s=500, marker='*',
                edgecolor='black', linewidth=2, label="3. Edited CLIP")

    plt.scatter(idx_er[0], idx_er[1], c='orange', s=500, marker='*',
                edgecolor='black', linewidth=2, label="4. Decoded Reconstructed (edited WM)")

    # 3) 연결선
    plt.plot([idx_o[0], idx_or[0]], [idx_o[1], idx_or[1]], 'cyan',   linestyle='--', linewidth=2)
    plt.plot([idx_o[0], idx_e[0]],  [idx_o[1], idx_e[1]],  'red',    linestyle='--', linewidth=2)
    plt.plot([idx_e[0], idx_er[0]], [idx_e[1], idx_er[1]], 'orange', linestyle='--', linewidth=2)
    plt.plot([idx_or[0], idx_er[0]], [idx_or[1], idx_er[1]], 'purple', linestyle=':', linewidth=1.5)

    plt.title(f"t-SNE for {fname}", fontsize=18, fontweight='bold')
    plt.grid(alpha=0.3, linestyle="--")
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

## 9. 최종 요약

In [ ]:
# ============================================================================
# 최종 요약 리포트
# ============================================================================
print("="*80)
print("🎯 Semantic Watermark Inference Pipeline 최종 요약")
print("="*80)

print("\n[1] 개별 이미지 기반 통계 복원")
print("  • 방식: 각 이미지의 원본 latent 평균(μ)과 표준편차(σ) 사용")
print("  • 복원: watermark → latent' = latent * σ + μ")
print("  • 효과: 개별 이미지 분포에 맞춘 맞춤형 복원")

# 2) 워터마크 삽입 결과 요약
if 'insertion_results' in locals():
    success_insert = sum(1 for r in insertion_results if r['status'] == 'success')
    print("\n[2] 워터마크 삽입 결과")
    print(f"  • 성공: {success_insert}장")

# 3) 추출 결과 요약
if 'extraction_results' in locals():
    success_extract = [r for r in extraction_results if r['status'] == 'success']
    if len(success_extract) > 0:
        print("\n[3] 워터마크 추출 및 의미 복원")
        print(f"  • 성공: {len(success_extract)}장")
        print(f"  • 원본 vs 워터마크 복원: {np.mean([r['cosine_original_vs_watermark'] for r in success_extract]):.4f}")
        print(f"  • 원본 vs 편집된 이미지: {np.mean([r['cosine_original_vs_edited'] for r in success_extract]):.4f}")
        print(f"  • 워터마크 복원 vs 편집: {np.mean([r['cosine_watermark_vs_edited'] for r in success_extract]):.4f}")

# 4) Training Cluster Summary
if 'all_embeddings' in locals() and all_embeddings is not None:
    print("\n[4] Training 클러스터 분석")
    print(f"  • Training 샘플 수: {len(all_embeddings)}개")
    print("  • Training embedding 기반 t-SNE 시각화 완료")

# 5) 저장된 파일 목록
print("\n[5] 저장된 파일")
saved_files = [
    ('training_clip_embeddings.pkl', 'Training CLIP embeddings'),
    ('insertion_results.pkl', '워터마크 삽입 결과'),
    ('extraction_results.pkl', '워터마크 추출 결과'),
]

for filename, desc in saved_files:
    file_path = os.path.join(BASE_DIR if 'training_clip' in filename else DATASET_DIR, filename)
    if os.path.exists(file_path):
        print(f"  ✓ {filename}: {desc}")

print("\n" + "="*80)
print("✅ 모든 파이프라인 실행 완료!")
print("="*80)